In [1]:
import subprocess, time

# Instalar zstd (necessário para o Ollama)
!apt-get update && apt-get install -y zstd -qq > /dev/null

# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Instalar dependências
!pip install ollama chromadb pymupdf -q

# Iniciar servidor Ollama em background
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# Baixar modelos
!!ollama pull llama3.1:8b
!ollama pull nomic-embed-text

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 https://cli.github.com/packages stable InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Pack

In [2]:
from google.colab import files

print("Faça upload dos 3 PDFs do projeto:")
print("1. Purdue - Failures in spacecraft systems")
print("2. NASA - Mars Surface Power Decision")
print("3. NASA - Capítulo 3.0 Potência")

arquivos_upload = files.upload()
nomes_pdfs = list(arquivos_upload.keys())
print(f"\nArquivos recebidos: {nomes_pdfs}")

Faça upload dos 3 PDFs do projeto:
1. Purdue - Failures in spacecraft systems
2. NASA - Mars Surface Power Decision
3. NASA - Capítulo 3.0 Potência


Saving 3.0 Potência - NASA.pdf to 3.0 Potência - NASA.pdf
Saving acr24-mars-surface-power-decision.pdf to acr24-mars-surface-power-decision.pdf
Saving Failures_in_spacecraft_systems__An_analysis_from_the_perspective_of_decision_making.pdf to Failures_in_spacecraft_systems__An_analysis_from_the_perspective_of_decision_making.pdf

Arquivos recebidos: ['3.0 Potência - NASA.pdf', 'acr24-mars-surface-power-decision.pdf', 'Failures_in_spacecraft_systems__An_analysis_from_the_perspective_of_decision_making.pdf']


In [3]:
import fitz  # pymupdf

def extrair_texto_pdf(caminho_pdf):
    documento = fitz.open(caminho_pdf)
    texto_completo = ""
    for pagina in documento:
        texto_completo += pagina.get_text()
    documento.close()
    return texto_completo

def criar_chunks(texto, tamanho_chunk=500, sobreposicao=100):
    palavras = texto.split()
    chunks = []
    inicio = 0

    while inicio < len(palavras):
        fim = inicio + tamanho_chunk
        chunk = " ".join(palavras[inicio:fim])
        chunks.append(chunk)
        inicio += tamanho_chunk - sobreposicao

    return chunks

# Processar cada PDF
todos_chunks = []
metadados_chunks = []

for nome_pdf in nomes_pdfs:
    print(f"Processando: {nome_pdf}")
    texto = extrair_texto_pdf(nome_pdf)
    chunks = criar_chunks(texto)

    for indice, chunk in enumerate(chunks):
        todos_chunks.append(chunk)
        metadados_chunks.append({
            "fonte": nome_pdf,
            "chunk_indice": indice,
        })

    print(f"  → {len(chunks)} chunks criados")

print(f"\nTotal de chunks: {len(todos_chunks)}")

Processando: 3.0 Potência - NASA.pdf
  → 32 chunks criados
Processando: acr24-mars-surface-power-decision.pdf
  → 6 chunks criados
Processando: Failures_in_spacecraft_systems__An_analysis_from_the_perspective_of_decision_making.pdf
  → 41 chunks criados

Total de chunks: 79


In [4]:
import chromadb
import ollama

cliente_chromadb = chromadb.Client()

colecao = cliente_chromadb.create_collection(
    name="mission_control_docs",
)

print("Gerando embeddings e armazenando chunks...")

for indice, chunk in enumerate(todos_chunks):
    resposta_embedding = ollama.embed(
        model="nomic-embed-text",
        input=chunk,
    )

    colecao.add(
        ids=[f"chunk_{indice}"],
        embeddings=[resposta_embedding["embeddings"][0]],
        documents=[chunk],
        metadatas=[metadados_chunks[indice]],
    )

    if (indice + 1) % 50 == 0:
        print(f"  → {indice + 1}/{len(todos_chunks)} chunks processados")

print(f"\nVector store criado com {colecao.count()} chunks.")

Gerando embeddings e armazenando chunks...
  → 50/79 chunks processados

Vector store criado com 79 chunks.


In [5]:
def buscar_contexto(pergunta, quantidade_resultados=5):
    embedding_pergunta = ollama.embed(
        model="nomic-embed-text",
        input=pergunta,
    )

    resultados = colecao.query(
        query_embeddings=[embedding_pergunta["embeddings"][0]],
        n_results=quantidade_resultados,
    )

    contextos = []
    for indice in range(len(resultados["documents"][0])):
        contextos.append({
            "texto": resultados["documents"][0][indice],
            "fonte": resultados["metadatas"][0][indice]["fonte"],
            "distancia": resultados["distances"][0][indice],
        })

    return contextos

In [6]:
def recomendar_acao(evento_nome, categoria, severidade, atributos_afetados, opcoes_resposta):

    pergunta_busca = (
        f"spacecraft {categoria} failure: {evento_nome}. "
        f"Affected systems: {', '.join(atributos_afetados)}. "
        f"Root cause analysis and recommended corrective action."
    )

    contextos = buscar_contexto(pergunta_busca, quantidade_resultados=2)

    contexto_formatado = ""
    fontes_usadas = set()
    for contexto in contextos:
        contexto_formatado += contexto["texto"] + "\n\n"

    opcoes_formatadas = ""
    for indice, opcao in enumerate(opcoes_resposta):
        opcoes_formatadas += f"  [{indice + 1}] {opcao['descricao']})\n"

    prompt_sistema = (
        "You are the AI system of a spacecraft. "
        "When an event occurs, you must choose exactly ONE option from the list provided. "
        "Reply ONLY in this format: [number] - [option name]. [one sentence why]. "
        "Nothing else. No introductions, no lists, no explanations beyond one sentence."
        "Answer in brazilian portuguese."
    )

    prompt_usuario = (
        f"Event: {evento_nome} (severity {severidade}/5)\n"
        f"Affected: {', '.join(atributos_afetados)}\n\n"
        f"Options:\n{opcoes_formatadas}\n"
        f"Technical reference:\n{contexto_formatado[:500]}\n\n"
        f"Choose ONE option."
    )

    resposta = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario},
        ],
    )

    return {
        "recomendacao": resposta["message"]["content"],
    }

In [8]:
import subprocess, time

# Ensure Ollama server is running
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# Simula o evento "Falha no motor principal" do EventEngine
resultado = recomendar_acao(
    evento_nome="Falha no motor principal",
    categoria="Engine",
    severidade=4,
    atributos_afetados=["fuel"],
    opcoes_resposta=[
        {"descricao": "Isolar motor secundário"},
        {"descricao": "Reduzir empuxo"},
        {"descricao": "Não agir"},
    ],
)

print("=" * 70)
print("  RECOMENDAÇÃO DA IA")
print("=" * 70)
print(resultado["recomendacao"])

INFO:werkzeug:127.0.0.1 - - [07/Jun/2026 19:23:07] "POST /recomendar HTTP/1.1" 200 -


  RECOMENDAÇÃO DA IA
[1] Isolar motor secundário) - A segurança da nave depende que o sistema de propulsão continue funcionando.


In [9]:
import subprocess, time

# Ensure Ollama server is running
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

CATALOGO_TESTE = [
    {"nome": "Falha no motor principal",       "categoria": "Engine",        "severidade": 4, "atributos": ["fuel"],                 "opcoes": [{"descricao": "Isolar motor secundário"}, {"descricao": "Reduzir empuxo"}, {"descricao": "Não agir"}]},
    {"nome": "Vazamento de propelente",        "categoria": "Engine",        "severidade": 3, "atributos": ["fuel"],                 "opcoes": [{"descricao": "Selar válvula de contenção"}, {"descricao": "Redirecionar fluxo"}, {"descricao": "Não agir"}]},
    {"nome": "Falha no painel solar",          "categoria": "Power",         "severidade": 3, "atributos": ["battery"],              "opcoes": [{"descricao": "Reorientar painel reserva"}, {"descricao": "Desligar sistemas não essenciais"}, {"descricao": "Não agir"}]},
    {"nome": "Perda de conexão com os sistemas de comunicação da terra",     "categoria": "Communication", "severidade": 3, "atributos": ["signal"],               "opcoes": [{"descricao": "Reorientar antena direcional"}, {"descricao": "Aumentar potência de transmissão"}, {"descricao": "Não agir"}]},
    {"nome": "Falha no controle de atitude",   "categoria": "AD&C",          "severidade": 3, "atributos": ["signal", "fuel"],       "opcoes": [{"descricao": "Ativar sistema de controle redundante"}, {"descricao": "Correção manual de trajetória"}, {"descricao": "Não agir"}]},
    {"nome": "Erro no software de navegação",  "categoria": "Programming",   "severidade": 4, "atributos": ["fuel", "signal"],       "opcoes": [{"descricao": "Reiniciar sistema de navegação"}, {"descricao": "Aplicar patch manual"}, {"descricao": "Não agir"}]},
]

for evento in CATALOGO_TESTE:
    resultado = recomendar_acao(
        evento_nome=evento["nome"],
        categoria=evento["categoria"],
        severidade=evento["severidade"],
        atributos_afetados=evento["atributos"],
        opcoes_resposta=evento["opcoes"],
    )

    print("=" * 70)
    print(f"  EVENTO: {evento['nome']}")
    print("=" * 70)
    print(resultado["recomendacao"])
    print()

  EVENTO: Falha no motor principal
[2] Reduzir empuxo) - Reduzir o empuxo ajudará a manter estabilidade e segurança no voo enquanto identificamos as causas da falha do motor principal.

  EVENTO: Vazamento de propelente
[1] Selar válvula de contenção) - Isolar o vazamento imediatamente para evitar maior dano ao sistema.

  EVENTO: Falha no painel solar
[2] Desligar sistemas não essenciais) - É necessário reduzir a carga no sistema para evitar sobrecarga e garantir que a bateria se recupere.

  EVENTO: Perda de conexão com os sistemas de comunicação da terra
[2] Aumentar potência de transmissão) - Poder aumentar a probabilidade de se reestabelecer a conexão com os sistemas de comunicação da Terra.

  EVENTO: Falha no controle de atitude
[1] Ativar sistema de controle redundante) - É a melhor opção para garantir que o veículo se mantenha estável e seguro até que os problemas possam ser mais completamente diagnosticados e corrigidos.

  EVENTO: Erro no software de navegação
[1] Reiniciar 

In [7]:
# Instalar dependências do servidor
!pip install flask pyngrok -q
from google.colab import userdata

from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading

app = Flask(__name__)

@app.route("/recomendar", methods=["POST"])
def endpoint_recomendar():
    dados = request.json

    resultado = recomendar_acao(
        evento_nome=dados["evento_nome"],
        categoria=dados["categoria"],
        severidade=dados["severidade"],
        atributos_afetados=dados["atributos_afetados"],
        opcoes_resposta=dados["opcoes_resposta"],
    )

    return jsonify({"recomendacao": resultado})

# Expor via ngrok
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))
tunnel = ngrok.connect(5000, domain="alienable-chef-pueblo.ngrok-free.dev")
print(f"URL fixa: {tunnel.public_url}")

# Rodar servidor em background
threading.Thread(target=lambda: app.run(port=5000)).start()

URL fixa: https://alienable-chef-pueblo.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
